# Toy BNN — predictive inspection

Loads `.pt` files from `results/toy_bnns/` and plots the posterior predictive mean ± 2σ against the true function and training points, for each dataset and sampler.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.gpu_friendly.scripts.toy_bnn_grid import build_target, DATASET_CONFIGS, BNNConfig
from sazz.gpu_friendly.models.neural_networks import FFN

torch.set_default_dtype(torch.float64)

RESULTS_DIR = Path("results/grid/toy_bnns")
DATA_DIR    = Path("datasets/toy_1d")

SAMPLER_STYLE = {
    "grid_boomerang":           ("C0", "Grid Boomerang"),
    "grid_sticky_boomerang":    ("C1", "Grid Sticky Boomerang"),
    "grid_zigzag":              ("C2", "Grid Zig Zag"),
    "grid_sticky_zigzag":       ("C3", "Grid Sticky Zig Zag"),
    "nuts":                     ("C4", "NUTS"),
}

# Discover which (dataset, split) directories have results
available = sorted(
    (p.parent.parent.name, int(p.parent.name.split("_")[1]))
    for p in RESULTS_DIR.glob("*/split_*/grid_boomerang.pt")
)
print("Available (dataset, split):", available)


In [ ]:
import torch
import numpy as np
from pathlib import Path

DATA_DIR = Path("datasets/toy_1d")

for ds in ["hernandez"]:
    d = torch.load(DATA_DIR / f"{ds}.pt", weights_only=False)
    x_raw = d["x_train_raw"]
    y_raw = d["y_train_raw"]
    print(f"=== {ds} ===")
    print(f"  n_train={d['n_train']}  x range=[{x_raw.min():.2f}, {x_raw.max():.2f}]  "
          f"x_mean={d['x_mean']:.3f}  x_std={d['x_std']:.3f}")
    print(f"  y range=[{y_raw.min():.2f}, {y_raw.max():.2f}]  "
          f"y_mean={d['y_mean']:.3f}  y_std={d['y_std']:.3f}")
    print(f"  noise_std_true={d['noise_std_true']:.4f}  "
          f"noise_std_standardised={d['noise_std']:.4f}  "
          f"signal/noise (y_std/noise_true)={d['y_std']/d['noise_std_true']:.2f}")


In [ ]:
def load_split(dataset: str, split_id: int):
    """Load dataset dict and all sampler payloads for one (dataset, split)."""
    data = torch.load(DATA_DIR / f"{dataset}.pt", weights_only=False)
    split_dir = RESULTS_DIR / dataset / f"split_{split_id:02d}"
    runs = {
        p.stem: torch.load(p, weights_only=False)
        for p in sorted(split_dir.glob("*.pt"))
    }
    return data, runs


def make_bm_from_payload(payload, data, dataset: str):
    cfg = BNNConfig(**DATASET_CONFIGS[dataset], noise_std=data["noise_std"])
    bm, x_ref, Sigma_inv = build_target(data, cfg)
    return bm


@torch.no_grad()
def predictive_summary(samples: torch.Tensor, bm, payload: dict, data, x_grid: np.ndarray):
    x_mean, x_std = float(data["x_mean"]), float(data["x_std"])
    y_mean, y_std = float(data["y_mean"]), float(data["y_std"])

    X_grid = torch.tensor(x_grid[:, None], dtype=torch.float64)
    X_grid_std = (X_grid - x_mean) / x_std

    if bm.learns_noise:
        weight_samples = samples[:, :-1]
        noise_std_std_scale = float(samples[:, -1].exp().mean())  # mean sigma, standardised scale
    else:
        weight_samples = samples
        noise_std_std_scale = float(payload["noise_std"])  # fixed, already standardised scale

    preds = torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_grid_std,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, n_grid], standardised y

    mean_std = preds.mean(0).numpy()
    epi_std  = preds.std(0).numpy()

    mean_orig  = mean_std * y_std + y_mean
    epi_orig   = epi_std * y_std
    noise_orig = noise_std_std_scale * y_std
    total_orig = np.sqrt(epi_orig**2 + noise_orig**2)
    return mean_orig, epi_orig, total_orig


In [ ]:
for dataset, split_id in available:
    data, runs = load_split(dataset, split_id)
    bm = make_bm_from_payload(next(iter(runs.values())), data, dataset)

    x_test  = data["x_test_raw"]
    y_true  = data["y_test_clean_raw"]
    x_train = data["x_train_raw"]
    y_train = data["y_train_raw"]

    if dataset == "hernandez":
        x_grid = np.sort(x_train)
    else:
        x_grid = np.sort(x_test)

    sort_idx = np.argsort(x_test)
    x_sorted = x_test[sort_idx]
    y_sorted = y_true[sort_idx]

    sampler_keys = [k for k in SAMPLER_STYLE if k in runs]
    n_cols = len(sampler_keys)

    fig, axes = plt.subplots(1, n_cols, figsize=(3.8 * n_cols, 3.6),
                              sharey=True, squeeze=False)

    for col, key in enumerate(sampler_keys):
        ax = axes[0, col]
        color, label = SAMPLER_STYLE[key]

        payload = runs[key]
        samples = payload["samples"]  # Tensor [S, D]
        mean, epi, total = predictive_summary(samples, bm, payload, data, x_grid)

        ax.fill_between(x_grid, mean - 2 * total, mean + 2 * total,
                         color=color, alpha=0.22, linewidth=0)
        ax.fill_between(x_grid, mean - 2 * epi, mean + 2 * epi,
                         color=color, alpha=0.20, linewidth=0)

        ax.plot(x_sorted, y_sorted, color="black", lw=1.4,
                alpha=0.8, zorder=3, label="true function")
        ax.plot(x_grid, mean, color=color, lw=1.8, zorder=4,
                label="predictive mean")
        ax.scatter(x_train, y_train, marker="x", color="black",
                   s=40, linewidths=1.2, zorder=5, label="training data")

        ax.set_title(label, fontsize=10)
        ax.set_xlabel("x", fontsize=9)
        if col == 0:
            ax.set_ylabel("y", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.grid(alpha=0.15, linewidth=0.5)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(handles),
               frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.04))

    fig.suptitle(f"{dataset}  (split {split_id})", fontsize=11, y=1.01)
    fig.tight_layout()
    plt.show()

## Predictive metrics

RMSE, NLL, CRPS, and 90%/95% coverage on the held-out (noisy) test set `X_test`/`y_test` -- not the dense plotting grid above, which uses the clean function for visualization only. Same formulas as `grid_uci_investigate.ipynb`. `ESS/grad` (ESS per gradient evaluation) is included since it's comparable across PDMP/NUTS despite their differing iteration/draw units; see the diagnostics section below for raw gradient-eval throughput.

In [ ]:
import math
import pandas as pd
from torch.distributions import Normal

from sazz.utils.metrics import ess_per_coord


@torch.no_grad()
def predict_on_test(samples: torch.Tensor, bm, data) -> tuple[torch.Tensor, torch.Tensor, float]:
    """Returns (mean_pred, epist_std, noise_std) on the standardised test set."""
    X_test = data["X_test"].to(dtype=torch.float64)

    if bm.learns_noise:
        weight_samples = samples[:, :-1]
        noise_std = float(samples[:, -1].exp().mean())
    else:
        weight_samples = samples
        noise_std = float(data["noise_std"])

    preds = torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_test,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, N]
    return preds.mean(0), preds.std(0), noise_std


def compute_rmse(y_true, mean_pred, y_std):
    return float(((mean_pred - y_true) ** 2).mean().sqrt()) * y_std


def compute_nll(y_true, mean_pred, total_std, y_std):
    ll = (-0.5 * ((y_true - mean_pred) / total_std) ** 2
          - total_std.log() - 0.5 * math.log(2 * math.pi)).mean()
    return float(-ll + math.log(y_std))


def compute_crps(y_true, mean_pred, total_std, y_std):
    d = Normal(0.0, 1.0)
    sigma = total_std * y_std
    z = (y_true * y_std - mean_pred * y_std) / sigma
    return float((sigma * (z * (2 * d.cdf(z) - 1) + 2 * d.log_prob(z).exp()
                            - 1 / math.sqrt(math.pi))).mean())


def compute_coverage(y_true, mean_pred, total_std, level=0.9):
    z = Normal(0.0, 1.0).icdf(torch.tensor(0.5 + level / 2))
    return float(((y_true - mean_pred).abs() <= z * total_std).float().mean())


metric_records = []

for dataset, split_id in available:
    data, runs = load_split(dataset, split_id)
    bm = make_bm_from_payload(next(iter(runs.values())), data, dataset)

    y_test = data["y_test"].to(dtype=torch.float64)
    y_std  = float(data["y_std"])

    for key in SAMPLER_STYLE:
        if key not in runs:
            continue
        payload = runs[key]
        samples = payload["samples"]
        mean_pred, epist_std, noise_std = predict_on_test(samples, bm, data)
        total_std = (epist_std ** 2 + noise_std ** 2).sqrt()

        weight_samples = samples[:, :-1] if bm.learns_noise else samples
        ess = ess_per_coord(weight_samples)
        grad_evals = payload.get("gradient_evals")

        metric_records.append({
            "dataset":  dataset,
            "split_id": split_id,
            "sampler":  SAMPLER_STYLE[key][1],
            "RMSE":     compute_rmse(y_test, mean_pred, y_std),
            "NLL":      compute_nll(y_test, mean_pred, total_std, y_std),
            "CRPS":     compute_crps(y_test, mean_pred, total_std, y_std),
            "Cov 90%":  compute_coverage(y_test, mean_pred, total_std, 0.90),
            "Cov 95%":  compute_coverage(y_test, mean_pred, total_std, 0.95),
            "ESS min":  float(ess.min()),
            # ESS/gradient-eval: comparable across PDMP/NUTS despite their
            # differing "iteration"/"draw" units (see diagnostics section).
            "ESS/grad": (float(ess.min()) / grad_evals) if grad_evals else float("nan"),
        })

metrics_df = pd.DataFrame(metric_records).set_index(["dataset", "split_id", "sampler"])

In [ ]:
def highlight_min_per_group(col):
    out = pd.Series("", index=col.index)
    for (dataset, split_id), group in col.groupby(level=("dataset", "split_id")):
        out.loc[group.idxmin()] = "background-color: #68dc0f"
    return out


def highlight_coverage(col):
    level = 0.90 if "90" in col.name else 0.95
    out = pd.Series("", index=col.index)
    for (dataset, split_id), group in col.groupby(level=("dataset", "split_id")):
        best = (group - level).abs().idxmin()
        out.loc[best] = "background-color: #68dc0f"
    return out


metrics_df.style \
    .apply(highlight_min_per_group, subset=["RMSE", "NLL", "CRPS"]) \
    .apply(highlight_coverage, subset=["Cov 90%", "Cov 95%"]) \
    .format(precision=4, na_rep="n/a")

## Sampler cost diagnostics (all samplers, all splits)

Compute-cost comparison across every `(dataset, split)` and sampler, in gradient evaluations of the log-target -- the common currency across PDMP "skeleton events" and NUTS "draws", which otherwise aren't comparable units. `t_max` columns are grid-sampler-only (NUTS has no adaptive horizon). `Grad evals`/`grid_t_max_log` are only present in `.pt` files saved after the runner scripts started persisting them -- older runs show `n/a`; re-run to backfill.

In [ ]:
diag_records = []

for dataset, split_id in available:
    _, runs = load_split(dataset, split_id)

    for key in SAMPLER_STYLE:
        if key not in runs:
            continue
        r = runs[key]

        n_events   = r.get("n_events")
        elapsed    = r.get("elapsed_sec")
        grad_evals = r.get("gradient_evals")
        t_max_log  = r.get("grid_t_max_log")
        bound_viol = r.get("bound_violations")

        diag_records.append({
            "dataset":          dataset,
            "split_id":         split_id,
            "sampler":          SAMPLER_STYLE[key][1],
            "Events/s":         (n_events / elapsed) if (n_events and elapsed) else float("nan"),
            "Grad evals/s":     (grad_evals / elapsed) if (grad_evals and elapsed) else float("nan"),
            "Grad evals/event": (grad_evals / n_events) if (grad_evals and n_events) else float("nan"),
            "t_max mean":       float(np.mean(t_max_log)) if t_max_log else float("nan"),
            "t_max min":        float(np.min(t_max_log)) if t_max_log else float("nan"),
            "t_max max":        float(np.max(t_max_log)) if t_max_log else float("nan"),
            "Bound violations": bound_viol if bound_viol is not None else float("nan"),
            "Time (s)":         elapsed if elapsed is not None else float("nan"),
        })

diag_df = pd.DataFrame(diag_records).set_index(["dataset", "split_id", "sampler"])
diag_df.style.format(precision=4, na_rep="n/a")